# Training on Kaggle

Trains MammoSeg on one or more cross-validation folds on a free Kaggle GPU. It
installs the package, rebuilds the derived data from the INbreast dataset and runs
`mammo-seg train` for each fold. Research use only: this is not a medical device.

**Once per notebook (settings panel):**
- *Add input*: the INbreast dataset.
- Accelerator: *GPU T4* (or P100). Choose it before running anything: switching it
  later restarts the machine and removes the installed packages.
- Internet: *on* (the repository and the pretrained encoder weights are downloaded).

**Run** the four code cells in order; only the first one needs editing. Never put
`kaggle.json` or any other credential in a notebook.

**Sessions end** after at most about 12 hours, or earlier if idle. Training stops
cleanly before `MAX_MINUTES` and saves `last.pt` after every epoch. To keep the
results, save a version of the notebook: its output holds the `runs` folder. To
continue in a later session, add that output as an input of the notebook, set
`PREVIOUS_RUNS` to its `runs` folder and run the cells again: a fold with a
`last.pt` is resumed, and a finished fold returns at once.

Colab was not tried: the paths of this notebook are those of Kaggle.

In [ ]:
from pathlib import Path

FOLDS = [0]  # [0, 1, 2, 3, 4] for the whole cross-validation
MAX_MINUTES = 10 * 60  # for all folds together; training stops cleanly before this
# More options of `mammo-seg train`, for example "--epochs 2 --batch-size 2".
TRAIN_OPTIONS = (
    "--crop-size 512 --batch-size 8 --samples-per-epoch 512 "
    "--patience 0 --selection-metric with_mass.dice"
)
REPO_URL = "https://github.com/GiannisPapa97/mammo_seg.git"
REPO_REF = "main"  # branch or tag to install
PREVIOUS_RUNS = None  # for example Path("/kaggle/input/my-notebook/runs")

## 1. Setup

Installs the package (`.[cloud]` leaves the PyTorch of the notebook, a GPU build,
untouched), finds the dataset and rebuilds the derived data: manifest, patient-level
splits, masks and preprocessed images (a few minutes). A step whose output exists is
skipped. The repository is cloned to `repo`, not `mammo_seg`: a folder of that name
next to the notebook would hide the installed package from Python.

The cells below only run `mammo-seg`. A cell of your own that imports `mammo_seg`
or `scipy` in this session fails with a numpy error, because the install upgraded
numpy after the notebook had loaded it: restart the session once (*Run* → *Restart
session*; files and installed packages are kept), run the first cell and this one
again (a few seconds), and then use your cell.

In [ ]:
WORK = Path("/kaggle/working")
REPO = WORK / "repo"
PROCESSED = WORK / "data" / "processed"
RUNS = WORK / "runs"
MANIFEST = PROCESSED / "manifest.json"
SPLITS = PROCESSED / "splits.json"
MASKS = PROCESSED / "masks"
PREPROCESSED = PROCESSED / "preprocessed"

found = sorted(Path("/kaggle/input").rglob("INbreast.csv"))
assert found, "attach the INbreast dataset as an input of the notebook"
DATASET_DIR = found[0].parent
print("dataset:", DATASET_DIR)

!test -d {REPO} || git clone -q --branch {REPO_REF} {REPO_URL} {REPO}
!pip install -q "{REPO}[cloud]"
!nvidia-smi -L

# The code of every session: the time and the commit, kept with the runs.
RUNS.mkdir(parents=True, exist_ok=True)
VERSION = RUNS / "code_version.txt"
previous_version = Path(PREVIOUS_RUNS) / "code_version.txt" if PREVIOUS_RUNS else None
if previous_version and previous_version.exists() and not VERSION.exists():
    VERSION.write_text(previous_version.read_text())
!echo "$(date -u +%FT%TZ) $(git -C {REPO} rev-parse HEAD)" | tee -a {VERSION}

PROCESSED.mkdir(parents=True, exist_ok=True)
if not MANIFEST.exists():
    !mammo-seg manifest --dataset-dir "{DATASET_DIR}" --output {MANIFEST}
if not SPLITS.exists():
    !mammo-seg splits --folds 5 --test-fraction 0.2 \
        --manifest {MANIFEST} --output {SPLITS}
if not MASKS.exists():
    !mammo-seg masks --manifest {MANIFEST} --output-dir {MASKS}
if not (PREPROCESSED / "preprocessing.json").exists():
    !mammo-seg preprocess --manifest {MANIFEST} --dataset-dir "{DATASET_DIR}" \
        --masks-dir {MASKS} --output-dir {PREPROCESSED}

## 2. Train

One run per fold in `FOLDS`, in `runs/fold_<k>`: `best.pt` (the epoch with the best
Dice over all validation images; `--selection-metric` can choose another metric),
`last.pt` and `history.jsonl`. `MAX_MINUTES` is shared by the folds: each gets what
is left. `--device cuda` makes a notebook without a GPU an error instead of a slow
run on the CPU; add `--device cpu` to `TRAIN_OPTIONS` to override it.

In [ ]:
import shutil
import time

started = time.monotonic()
for fold in FOLDS:
    minutes = int(MAX_MINUTES - (time.monotonic() - started) / 60)
    if minutes < 1:
        print(f"fold {fold}: MAX_MINUTES is used up; run this cell in a new session")
        continue
    fold_dir = RUNS / f"fold_{fold}"
    previous = Path(PREVIOUS_RUNS) / f"fold_{fold}" if PREVIOUS_RUNS else None
    if previous and previous.exists() and not fold_dir.exists():
        shutil.copytree(previous, fold_dir)
    resume = "--resume" if (fold_dir / "last.pt").exists() else ""
    print(f"=== fold {fold}: {'resuming' if resume else 'new run'}, {minutes} min left")
    !mammo-seg train --manifest {MANIFEST} --splits {SPLITS} \
        --preprocessed-dir {PREPROCESSED} --output-dir {RUNS} --device cuda \
        --num-workers 2 {TRAIN_OPTIONS} --fold {fold} --max-minutes {minutes} {resume}

## 3. Results

The best epoch of each fold by the selection metric (Dice over all validation
images unless changed), with the Dice on the images with a mass, their sensitivity
and the share of images without a mass on which anything was predicted. `n/a` means
the score was undefined. Save a version of the notebook to keep `runs`.

In [ ]:
import json


def show(value):
    return "n/a" if value is None else f"{value:.3f}"


selected = []
for fold in FOLDS:
    history = RUNS / f"fold_{fold}" / "history.jsonl"
    lines = history.read_text().splitlines() if history.exists() else []
    rows = [json.loads(line) for line in lines]
    scored = [row for row in rows if row["selection_value"] is not None]
    if not scored:
        print(f"fold {fold}: {len(rows)} epochs, none with a defined score")
        continue
    best = max(scored, key=lambda row: row["selection_value"])
    validation = best["validation"]
    selected.append(best["selection_value"])
    print(
        f"fold {fold}: {len(rows)} epochs, best epoch {best['epoch']}: "
        f"score {show(best['selection_value'])}, "
        f"Dice {show(validation['all_images']['dice'])}, "
        f"with mass {show(validation['with_mass']['dice'])} "
        f"(sensitivity {show(validation['with_mass']['sensitivity'])}), "
        f"false alarms {show(validation['without_mass']['false_alarm_rate'])}"
    )
if len(selected) > 1:
    print(f"mean score over {len(selected)} folds: {sum(selected) / len(selected):.3f}")